In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import Window
from pyspark.sql.functions import *

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("PostgresConnector") \
    .config("spark.jars","/usr/lib/spark/jars/postgresql-42.7.4.jar")\
    .getOrCreate()


In [ ]:
# Question 41: Generate a report of total hires per month, filling months with 0 hires.
# Concept: Time Series Generation (SQL Recursive vs PySpark Range Join)
employee_df = spark.read.jdbc(url="jdbc:postgresql://localhost:5432/employees",table="employees.employee",
                         properties={"user": "vagrant", "password": "vagrant","driver": "org.postgresql.Driver"})

department_df = spark.read.jdbc(url="jdbc:postgresql://localhost:5432/employees",table="employees.department",
                         properties={"user": "vagrant", "password": "vagrant","driver": "org.postgresql.Driver"})

department_employee_df = spark.read.jdbc(url="jdbc:postgresql://localhost:5432/employees",table="employees.department_employee",
                         properties={"user": "vagrant", "password": "vagrant","driver": "org.postgresql.Driver"})
# Solution:
import pandas as pd

# 1. Generate Month Series
min_date = employee_df.agg(min("hire_date")).collect()[0][0].replace(day=1)
max_date = employee_df.agg(max("hire_date")).collect()[0][0].replace(day=1)
date_range = pd.date_range(start=min_date, end=max_date, freq='MS')
months_df = spark.createDataFrame(date_range.to_frame(index=False), DateType()).toDF("month_start")

# 2. Prepare hires data
hires_df = employee_df.withColumn("month_start", trunc("hire_date", "month")) \
                       .groupBy("month_start") \
                       .count()

# 3. Left Join to fill zeros
months_df.join(hires_df, "month_start", "left") \
         .na.fill(0, subset=["count"]) \
         .orderBy("month_start") \
         .show()
